# Day 2 — Hidden states

This walkthrough starts with model loading and an inference-only smoke run. Extracting the first-token representation belongs to D2-02.

In [2]:
import sys
from pathlib import Path

import torch

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from transformers_learning import (
    DEFAULT_MODEL_NAME,
    extract_first_token_representation,
    get_embeddings,
    load_model,
    load_tokenizer,
    similarity,
)

## D2-01 — Model inference

The tokenizer and model use one checkpoint name. `eval()` disables training-only behavior such as dropout, while `torch.no_grad()` prevents PyTorch from building a gradient graph for this inference call.

In [3]:
model_name = DEFAULT_MODEL_NAME
tokenizer = load_tokenizer(model_name)
model = load_model(model_name)
device = next(model.parameters()).device

print(f"Model name: {model_name}")
print(f"Device: {device}")
print(f"Training mode: {model.training}")
print(model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model name: distilbert-base-uncased
Device: cpu
Training mode: False
DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)


In [4]:
text = "This movie was absolutely amazing!"
tokens = tokenizer(text, return_tensors="pt")
tokens = {name: tensor.to(device) for name, tensor in tokens.items()}

with torch.no_grad():
    gradients_enabled_during_inference = torch.is_grad_enabled()
    outputs = model(**tokens)

print(f"Output type: {type(outputs)}")
print(f"last_hidden_state shape: {tuple(outputs.last_hidden_state.shape)}")
print(f"Gradients enabled during inference: {gradients_enabled_during_inference}")

Output type: <class 'transformers.modeling_outputs.BaseModelOutput'>
last_hidden_state shape: (1, 8, 768)
Gradients enabled during inference: False


## D2-02 — First-token representation

`last_hidden_state` has shape `[batch, sequence, hidden]`: one contextual vector of `hidden` values for every token in every batch item. Selecting index `0` on the sequence axis produces `[batch, hidden]`. DistilBERT places `[CLS]` first, but its base model has no pooler and does not train this vector specifically as a universal sentence embedding.

In [5]:
last_hidden_state = outputs.last_hidden_state
first_token_representation = extract_first_token_representation(last_hidden_state)
batch_size, sequence_length, hidden_size = last_hidden_state.shape

print(f"last_hidden_state [batch, sequence, hidden]: {tuple(last_hidden_state.shape)}")
print(f"batch={batch_size}, sequence={sequence_length}, hidden={hidden_size}")
print(f"first-token representation [batch, hidden]: {tuple(first_token_representation.shape)}")
print(f"First five values: {first_token_representation[0, :5]}")

assert last_hidden_state.ndim == 3
assert first_token_representation.shape == (batch_size, hidden_size)

last_hidden_state [batch, sequence, hidden]: (1, 8, 768)
batch=1, sequence=8, hidden=768
first-token representation [batch, hidden]: (1, 768)
First five values: tensor([ 0.0682, -0.0685,  0.1918,  0.0139, -0.0544])


## D2-03 — Batched embeddings

`get_embeddings` preserves input order, runs inference without gradients, and returns one NumPy row per text. A batch size of three makes the final batch contain only one of the four examples. Empty input is rejected because its output width cannot be inferred without running the model.

In [6]:
texts = [
    "This movie was absolutely amazing!",
    "Terrible movie, waste of time.",
    "Pretty good, I liked it.",
    "Boring and too long.",
]

embeddings = get_embeddings(texts, tokenizer, model, batch_size=3)

print(f"Input texts: {len(texts)}")
print(f"Embeddings shape: {embeddings.shape}")
print(f"NumPy dtype: {embeddings.dtype}")
print(embeddings)
assert embeddings.shape == (len(texts), model.config.hidden_size)

Input texts: 4
Embeddings shape: (4, 768)
NumPy dtype: float32
[[ 0.0682218  -0.06854356  0.19183904 ... -0.18599822  0.39768797
   0.21738122]
 [ 0.0033977  -0.04019875 -0.00593721 ... -0.251224    0.3958779
   0.16264307]
 [-0.06282158 -0.12721898  0.04524297 ... -0.10008835  0.19173174
   0.24090998]
 [-0.05920587 -0.22124752  0.07999291 ... -0.05016062  0.36998263
   0.1531535 ]]


## D2-04 — Cosine similarity

Cosine similarity compares vector directions and is bounded by `[-1, 1]`. The pairs below probe similar positive wording, opposite sentiment within one topic, and an unrelated topic.

In [7]:
text_pairs = [
    ("similar positive wording", "Great movie!", "Amazing film!"),
    ("opposite sentiment", "Great movie!", "Terrible film!"),
    ("unrelated topic", "Great movie!", "The projector is made of metal."),
]

similarity_scores = []
for label, text1, text2 in text_pairs:
    score = similarity(text1, text2, tokenizer, model)
    similarity_scores.append(score)
    print(f"{label:25s}: {score:.3f}")

similar positive wording : 0.995
opposite sentiment       : 0.984
unrelated topic          : 0.909


These scores describe the geometry of first-token representations from base DistilBERT. Shared vocabulary, syntax, and topic can dominate the result, so a higher score is not a calibrated probability of matching sentiment and no particular ranking is guaranteed.

## D2-05 — Day 2 review

### Data flow and reusable API

| Boundary | Shape | Meaning | Reusable API |
|---|---|---|---|
| Raw input | `list[str]` | Texts in caller order | — |
| Tokenizer output | `[batch, sequence]` | Token IDs and attention mask | `tokenize_texts` |
| Model output | `[batch, sequence, hidden]` | Contextual vector for every token | `load_model` + model call |
| First-token slice | `[batch, hidden]` | Contextual representation at sequence index `0` | `extract_first_token_representation` |
| Batched result | `[number_of_texts, hidden]` | One NumPy row per input text | `get_embeddings` |
| Pair comparison | scalar `float` | Cosine similarity in `[-1, 1]` | `similarity` |

### Inference boundaries

- `model.eval()` disables training-only behavior such as dropout, but does not disable gradient tracking.
- `torch.no_grad()` disables graph construction only inside its context, but does not switch off dropout.
- Both are used for stable, memory-efficient inference. Moving the result to CPU and converting it to NumPy ends the PyTorch gradient path.
- Batching controls peak memory use; it does not change input order, and the final batch may be smaller.

### Limitations

- The first token is `[CLS]` for this tokenizer, but base DistilBERT has no pooler and does not optimize this vector as a universal sentence embedding.
- Cosine similarity measures vector direction, not sentiment agreement or a calibrated probability.
- Inputs are truncated to 128 tokens, so information after that boundary is discarded.
- No labels, train/test split, fitting, or model selection are used on Day 2, so there is no current data-leakage path. Those boundaries become essential when these embeddings feed a classifier.

In [8]:
assert model.training is False
assert gradients_enabled_during_inference is False
assert last_hidden_state.ndim == 3
assert first_token_representation.shape == (batch_size, hidden_size)
assert embeddings.shape == (len(texts), hidden_size)
assert all(isinstance(value, float) and -1.0 <= value <= 1.0 for value in similarity_scores)

print("Day 2 checkpoint passed.")

Day 2 checkpoint passed.
